### Extract open chromatin sequences and closed sequences into 270bp fasta
- 

In [9]:
import pandas as pd
import math
import yaml

# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf



In [ ]:

# open_ngn2_atac_peaks.columns[0:3]

Index([0, 1, 2], dtype='int64')

In [10]:
# /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/DIV14_WTC11-NGN2_ExN_ATAC-seq.idr0.05.bfilt.narrowPeak
open_ngn2_atac_peaks = pd.read_csv(config['files']['creating']['wtc11_ngn2_atac_narrowPeak'], sep="\t", header=None)
open_ngn2_atac_peaks # chrom, start, end name, score

# open_ngn2_atac_peaks[open_ngn2_atac_peaks.columns[0:3]].to_csv('/home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/wtc11_ngn2_atac_peak.bed', sep="\t", header=None, index=False)
open_ngn2_atac_peaks[open_ngn2_atac_peaks.columns[0:3]].to_csv(config['files']['creating']['wtc11_ngn2_atac_bed_no_name'], sep="\t", header=None, index=False)
# bedtools getfasta -fi genome.fa -bed open_chromatin.bed -fo open_chromatin.fa


In [ ]:
# !bedtools getfasta -fi /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa -bed /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/wtc11_ngn2_atac_peak.bed -fo /home/kisa/coding/80K_MPRA/modeling/open_chromatin_wtc11_ngn2.fa



### Make sequences 270 bp and remove N containing sequences
- start with N: 18 + 4 (overall number of rows => 22)

In [14]:
from Bio import SeqIO

def remove_n(sequence):
    return sequence.replace('N', '')

def extract_270bp_sequences(input_fasta, output_fasta):
    with open(output_fasta, 'w') as out_f:
        for record in SeqIO.parse(input_fasta, 'fasta'):
            sequence = str(record.seq)
            sequence = remove_n(sequence)
            for i in range(0, len(sequence) - 269, 270):
                subseq = sequence[i:i+270]
                out_f.write(f">{record.id}_{i}\n{subseq}\n")

extract_270bp_sequences('/home/kisa/coding/80K_MPRA/modeling/open_chromatin_wtc11_ngn2.fa', '/home/kisa/coding/80K_MPRA/modeling/open_chromatin_wtc11_ngn2_270bp.fa')
# extract_270bp_sequences('closed_chromatin.fa', 'closed_chromatin_270bp.fa')



### Get complement of the open chromatin sequences => closed chromatin

In [ ]:
# !bedtools complement -i /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/wtc11_ngn2_atac_peak_sorted.bed -g /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa.genome > /home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2.bed
# !bedtools getfasta -fi /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa -bed /home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2.bed -fo /home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2.fa


In [15]:
extract_270bp_sequences('/home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2.fa', '/home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2_270bp.fa')


In [22]:
open_chromatin_wtc11 = hf.fasta_to_dataframe('/home/kisa/coding/80K_MPRA/modeling/open_chromatin_wtc11_ngn2_270bp.fa')
open_chromatin_wtc11
# make all chars upper case
open_chromatin_wtc11['sequence'] = open_chromatin_wtc11['sequence'].str.upper()
open_chromatin_wtc11

# add column: is_open = True
open_chromatin_wtc11['is_open'] = True

open_chromatin_wtc11.to_csv('/home/kisa/coding/80K_MPRA/modeling/open_chromatin_wtc11_ngn2_270bp_outcome.tsv', sep="\t", index=False)


In [ ]:
# computed with python script /home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_element_analysis/scripts/add_outcome_to_chromatin_fa.py
closed_chromatin_wtc11 = hf.fasta_to_dataframe('/home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2_270bp.fa')
# make all chars upper case
closed_chromatin_wtc11['sequence'] = closed_chromatin_wtc11['sequence'].str.upper()
closed_chromatin_wtc11

# add column: is_open = True
closed_chromatin_wtc11['is_open'] = True

closed_chromatin_wtc11.to_csv('/home/kisa/coding/80K_MPRA/modeling/closed_chromatin_wtc11_ngn2_270bp_outcome.tsv', sep="\t", index=False)


KeyboardInterrupt: 